# eALS sur Spark — démonstration de bout en bout sur petites données

Notebook compagnon du rapport *Factorisation matricielle rapide pour la recommandation
en ligne avec rétroaction implicite (He et al., SIGIR 2016) — une implémentation Spark*.

Tout ce qui suit s'exécute sur **ml-100k** (943 utilisateurs, 1 152 articles, ~98k
interactions) et se termine en quelques minutes sur un portable. Son unique but est de
permettre au lecteur de vérifier que le code de `src/` fonctionne réellement : chaque
section indique **ce qu'elle prend en entrée** et **ce qu'elle affiche en sortie**.

Sommaire

1. Environnement et session Spark
2. Données brutes → filtrage 10-core → identifiants denses → découpage leave-one-out
3. Les poids dépendants de la popularité $c_i$ (Éq. 8)
4. Implémentation de référence (NumPy) — l'objectif $J$ décroît-il ?
5. Implémentation Spark RDD — obtient-on le même modèle ?
6. Contrôle d'exactitude : NumPy contre Spark RDD
7. Précision : eALS contre MostPopular contre ALS de MLlib
8. Une petite mesure de scalabilité (1/2/4 cœurs)

## 1. Environnement et session Spark

**Entrée** — rien d'autre que le JDK installé.

**Sortie** — les versions de Java, Spark et NumPy réellement utilisées.

`JAVA_HOME` doit pointer vers un JDK **17 ou 21**. Spark 4 ne s'exécute pas sur un
JDK 24 ou supérieur (`UnsupportedOperationException: getSubject is not supported`,
JEP 486). C'est de loin la façon la plus courante d'échouer avant même d'avoir écrit
une ligne d'algorithme.

In [1]:
import os, sys, subprocess, time
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
# One BLAS thread per task: Spark is the only parallelism we want to measure.
for v in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS",
          "VECLIB_MAXIMUM_THREADS","NUMEXPR_NUM_THREADS"):
    os.environ[v] = "1"
sys.path.insert(0, os.path.abspath("../src"))

import numpy as np, pandas as pd, pyspark
print(subprocess.run(["java","-version"], capture_output=True, text=True).stderr.splitlines()[0])
print("pyspark", pyspark.__version__, "| numpy", np.__version__)

openjdk version "17.0.20.1" 2026-08-18
pyspark 4.1.0 | numpy 2.3.5


In [2]:
from spark_utils import get_spark
spark = get_spark("eals-demo", cores=4, driver_mem="6g")
sc = spark.sparkContext
print(spark.version, "| master:", sc.master, "| default parallelism:", sc.defaultParallelism)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/27 18:08:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/27 18:08:29 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


4.1.0 | master: local[4] | default parallelism: 4


## 2. Des notes brutes à une matrice d'entraînement

**Entrée** — `data/raw/ml-100k/u.data`, le fichier MovieLens brut
(`utilisateur \t article \t note \t horodatage`).

**Sortie** — les statistiques du jeu de données et les premières lignes du découpage
d'entraînement.

Trois opérations ont lieu dans `data.prepare`, toutes en Spark :

* les notes sont **écartées** — la rétroaction implicite n'enregistre que le fait
  qu'une interaction a eu lieu, donc $r_{ui}=1$ ;
* le **filtrage 10-core** est appliqué de façon itérative, car retirer un utilisateur
  peut faire passer un article sous le seuil, et réciproquement ;
* les identifiants sont **ré-indexés de façon dense**, puis la dernière interaction de
  chaque utilisateur (au sens de l'horodatage) est mise de côté pour le test
  (leave-one-out).

In [ ]:
import data as D
stats = D.prepare(spark, "ml-100k", k=10)
pd.Series(stats).to_frame("value")

In [4]:
train = D.load_split(spark, "ml-100k", "train").cache()
test  = D.load_split(spark, "ml-100k", "test").cache()
M = train.agg({"u":"max"}).collect()[0][0] + 1
N = train.agg({"i":"max"}).collect()[0][0] + 1
print(f"M={M} users, N={N} items, |R|={train.count()} training, {test.count()} test")
train.show(5)

M=943 users, N=1152 items, |R|=97010 training, 943 test
+---+----+---------+
|  u|   i|       ts|
+---+----+---------+
|  0| 979|892870589|
|  0| 787|892387807|
|  0|1068|892015236|
|  0| 936|892015178|
|  0|1070|892015023|
+---+----+---------+
only showing top 5 rows


## 3. Le poids des cases vides dépendant de la popularité

**Entrée** — les fréquences des articles de l'ensemble d'entraînement.

**Sortie** — le vecteur $c_i$ de l'Éq. (8) et un aperçu de sa forme.

$$c_i = c_0\frac{f_i^{\alpha}}{\sum_j f_j^{\alpha}},\qquad f_i=\frac{|\mathcal R_i|}{\sum_j|\mathcal R_j|}$$

$\alpha=0$ redonne la pondération uniforme de Hu et al. (chaque case vide pèse
$c_0/N$) ; $\alpha=0,4$ fait qu'une case vide sur un grand succès compte plusieurs fois
plus qu'une case vide sur un titre confidentiel.

Deux choses méritent d'être lues dans la sortie. D'abord, $\sum_i c_i=c_0$ quelle que
soit la valeur de $\alpha$ : le paramètre **redistribue** le poids des cases vides, il
n'en ajoute pas. Ensuite, sur ce petit catalogue ($N=1152$), l'article le plus populaire
finit avec $c_i>1$, c'est-à-dire *davantage que le poids d'une interaction observée*.
Rien dans le modèle ne l'interdit, et c'est la réserve discutée en section 6.1 (S2) du
rapport.

In [5]:
from eals_local import item_confidence
counts = np.zeros(N)
for i, n in train.groupBy("i").count().collect():
    counts[i] = n
c04 = item_confidence(counts, c0=512, alpha=0.4)
c00 = item_confidence(counts, c0=512, alpha=0.0)
print(f"alpha=0.0 : c_i constant = {c00[0]:.4f}   (= c0/N = {512/N:.4f})")
print(f"alpha=0.4 : min={c04.min():.4f}  median={np.median(c04):.4f}  max={c04.max():.4f}"
      f"  ratio max/min = {c04.max()/c04.min():.0f}x")
print("sum c_i =", c04.sum().round(3), "= c0 in both cases")

alpha=0.0 : c_i constant = 0.4444   (= c0/N = 0.4444)
alpha=0.4 : min=0.1929  median=0.4109  max=1.0716  ratio max/min = 6x
sum c_i = 512.0 = c0 in both cases


## 4. Implémentation de référence (NumPy)

**Entrée** — les paires d'entraînement sous forme de deux tableaux d'entiers.

**Sortie** — la valeur de l'objectif $J$ (Éq. 7, évaluée via l'Éq. 14) après chaque
itération.

`eals_local.fit` est l'Algorithme 1 de l'article, écrit une seule fois et réutilisé tel
quel à l'intérieur de chaque tâche Spark. Si $J$ ne décroît pas de façon monotone, les
règles de mise à jour sont fausses.

In [6]:
import eals_local as L
pdf = train.toPandas()
u = pdf.u.to_numpy(np.int64); i = pdf.i.to_numpy(np.int64)

trace = []
t0 = time.perf_counter()
P_np, Q_np, c_np = L.fit(u, i, M, N, K=16, lam=0.01, c0=512, alpha=0.4,
                         iters=10, seed=42, trace=trace)
print(f"10 iterations in {time.perf_counter()-t0:.2f}s (single thread)")
pd.DataFrame({"iteration": range(1, 11), "objective J": np.round(trace, 2),
              "decrease": np.round(np.r_[np.nan, np.diff(trace)], 2)})

10 iterations in 0.45s (single thread)


,iteration,objective J,decrease
0,1,62499.47,NaN
1,2,44315.23,-18184.25
2,3,42182.27,-2132.95
3,4,41307.33,-874.95
4,5,40803.35,-503.98
5,6,40489.62,-313.74
6,7,40288.63,-200.99
7,8,40151.80,-136.83
8,9,40050.86,-100.93
9,10,39971.79,-79.07


## 5. Implémentation Spark RDD

**Entrée** — le même DataFrame d'entraînement, plus un nombre de partitions.

**Sortie** — le temps réel par itération, décomposé par phase.

Une itération se déroule ainsi : diffusion (broadcast) de $Q$ et de $S^q$ → une tâche
par bloc d'utilisateurs → réduction des contributions $P^{\top}P$ en $S^p$ → diffusion
de $P$ → une tâche par bloc d'articles → réduction des contributions
$\sum_i c_i q_iq_i^{\top}$ en $S^q$ pour le tour suivant.

Les deux caches $S$ ne coûtent donc rien de plus : chaque tâche renvoie la matrice de
Gram des lignes qu'elle vient d'écrire, et le driver n'a plus qu'à les additionner.

In [7]:
import eals_rdd as R
P_sp, Q_sp, c_sp, hist = R.fit(spark, train, M, N, K=16, lam=0.01, c0=512, alpha=0.4,
                               iters=10, partitions=8, seed=42)
h = pd.DataFrame(hist)[["iter","t_total","t_user","t_item","t_bcast","t_driver"]]
print(h.round(3).to_string(index=False))
print(f"\nmedian s/iteration: {h.t_total[1:].median():.3f}"
      f"   (broadcast+driver = {100*(h.t_bcast+h.t_driver)[1:].median()/h.t_total[1:].median():.1f}% of it)")

 iter  t_total  t_user  t_item  t_bcast  t_driver
    0    0.314   0.152   0.155    0.006     0.001
    1    0.339   0.171   0.159    0.009     0.001
    2    0.331   0.163   0.161    0.007     0.001
    3    0.342   0.163   0.170    0.009     0.001
    4    0.317   0.157   0.152    0.006     0.001
    5    0.350   0.178   0.163    0.008     0.001
    6    0.355   0.175   0.171    0.009     0.001
    7    0.360   0.177   0.174    0.008     0.001
    8    0.338   0.171   0.159    0.007     0.001
    9    0.346   0.171   0.165    0.009     0.001

median s/iteration: 0.342   (broadcast+driver = 2.6% of it)


## 6. Contrôle d'exactitude

**Entrée** — les modèles produits en 4 (référence NumPy) et en 5 (Spark RDD).

**Sortie** — le plus grand écart absolu entre les matrices de facteurs.

À $Q$ et $S^q$ fixés, les $M$ mises à jour d'utilisateurs portent sur des paramètres
disjoints. Répartir les utilisateurs entre plusieurs workers **ne peut donc pas**
changer le résultat : la parallélisation d'eALS est exacte, et non approchée. Le seul
écart admissible vient de l'ordre des sommations en virgule flottante, soit environ
$10^{-14}$. Tout écart plus grand est un bug.

In [ ]:
rows = [dict(implementation="spark-rdd",
             max_abs_dP=np.abs(P_sp - P_np).max(),
             max_abs_dQ=np.abs(Q_sp - Q_np).max())]
out = pd.DataFrame(rows)
print(out.to_string(index=False, float_format=lambda v: f"{v:.2e}"))
assert out[["max_abs_dP","max_abs_dQ"]].to_numpy().max() < 1e-10, "implementations disagree!"
print("\nPASS - both implementations produce the same model.")

## 7. Quelle est la qualité du modèle ?

**Entrée** — les matrices de facteurs et l'ensemble de test leave-one-out.

**Sortie** — HR@10/@100 et NDCG@10/@100.

L'article mis de côté est classé face à **tous** les articles que l'utilisateur n'a pas
vus pendant l'entraînement : c'est le protocole complet de l'article, et non la variante
économique à « 100 négatifs échantillonnés ».

`MostPopular` sert de plancher : un modèle incapable de le battre n'a rien appris.

In [9]:
import evaluate as EV, baselines as BL
res = {}
Pp, Qp = EV.most_popular_factors(train, M, N)
res["MostPopular"] = EV.rank_metrics(spark, Pp, Qp, train, test)
res["eALS (alpha=0.4)"] = EV.rank_metrics(spark, P_sp, Q_sp, train, test)

P0, Q0, _, _ = R.fit(spark, train, M, N, K=16, lam=0.01, c0=512, alpha=0.0,
                     iters=10, partitions=8, seed=42)
res["eALS (alpha=0, uniform)"] = EV.rank_metrics(spark, P0, Q0, train, test)

Pa, Qa, t_als = BL.fit_mllib_als(train, M, N, K=16, lam=0.01, iters=10, blocks=8)
res["MLlib ALS (Hu et al.)"] = EV.rank_metrics(spark, Pa, Qa, train, test)

pd.DataFrame(res).T.round(4)

,HR@10,HR@100,NDCG@10,NDCG@100
MostPopular,0.0456,0.2110,0.0237,0.0546
eALS (alpha=0.4),0.0764,0.4867,0.0379,0.1157
"eALS (alpha=0, uniform)",0.0732,0.4698,0.0334,0.1078
MLlib ALS (Hu et al.),0.0764,0.4719,0.0353,0.1087


## 8. Une petite mesure de scalabilité

**Entrée** — rien de neuf ; le même ensemble d'entraînement est réajusté avec 1, 2 puis
4 cœurs Spark. (Les courbes réelles du rapport vont jusqu'à 8 cœurs sur ml-10m, avec
3 répétitions entrelacées.)

**Sortie** — secondes par itération, accélération et efficacité.

Il faut s'attendre à une accélération **inférieure à 1** : sur ml-100k, ajouter des
cœurs rend eALS *plus lent*. Le calcul d'une itération ne dure ici que quelques dizaines
de millisecondes, alors que chaque bloc supplémentaire coûte à lui seul des dizaines de
millisecondes de surcoût Spark ; le surcoût croît donc plus vite que le travail ne
diminue.

Ce n'est ni un bug ni un résultat dissimulé : c'est exactement ce que montre la
Section 5.2 du rapport, où l'efficacité chute déjà nettement sur ml-1m et ne devient
acceptable que sur ml-10m, dix à cent fois plus gros que ce notebook.

In [10]:
spark.stop()
rows = []
for cores in (1, 2, 4):
    s = get_spark(f"scal-{cores}", cores=cores, driver_mem="4g")
    tr = D.load_split(s, "ml-100k", "train").cache()
    _, _, _, hh = R.fit(s, tr, M, N, K=16, iters=4, partitions=cores*2,
                        lam=0.01, c0=512, alpha=0.4)
    rows.append(dict(cores=cores, s_per_iter=float(np.median([x["t_total"] for x in hh[1:]]))))
    s.stop()
sc_df = pd.DataFrame(rows)
sc_df["speedup"] = sc_df.s_per_iter.iloc[0] / sc_df.s_per_iter
sc_df["efficiency"] = sc_df.speedup / sc_df.cores
sc_df.round(3)

,cores,s_per_iter,speedup,efficiency
0,1,0.340,1.000,1.000
1,2,0.414,0.822,0.411
2,4,0.545,0.625,0.156


## Ce que ce notebook a montré

1. le prétraitement reproduit un jeu implicite filtré en 10-core et le découpage
   leave-one-out de l'article ;
2. la référence NumPy fait décroître $J$ de façon monotone — les règles de mise à
   jour (12) et (13) sont correctement implémentées ;
3. l'implémentation Spark RDD retourne le **même modèle** que la référence NumPy à
   $10^{-14}$ près, ce qui confirme qu'eALS se parallélise exactement ;
4. eALS bat MostPopular avec une large marge et se situe au niveau de l'ALS de MLlib
   (légèrement au-dessus à @100) ;
5. sur des données aussi petites, Spark est *plus lent* que NumPy seul — ajouter des
   cœurs ralentit eALS, car le coût fixe par itération de Spark domine à $10^5$
   interactions.